  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 🚀 GraphCodeBERT Vulnerability Detection - Google Colab Training\n",
    "\n",
    "Notebook này giúp bạn train model phát hiện lỗ hổng bảo mật trong Java code trên Google Colab.\n",
    "\n",
    "## 📋 Các bước thực hiện:\n",
    "1. ✅ Setup environment & GPU\n",
    "2. 📂 Clone repository từ GitHub (đã có sẵn data và Python files)\n",
    "3. 📥 Install dependencies\n",
    "4. 🔍 Verify data\n",
    "5. 🎯 Tokenize data\n",
    "6. 🏋️ Train model\n",
    "7. 💾 Save & download model\n",
    "\n",
    "---\n",
    "\n",
    "**⚠️ Quan trọng**: \n",
    "- Nhớ enable GPU trong Runtime → Change runtime type → GPU (T4)\n",
    "- Thay `YOUR_GITHUB_REPO_URL` bằng link GitHub repo của bạn"
   ]
  },

## 1️⃣ Setup Environment & Check GPU

In [ ]:
import os
import sys

# Check GPU
!nvidia-smi

print("\n" + "="*70)
print("ENVIRONMENT INFO")
print("="*70)
print(f"Python version: {sys.version}")
print(f"Working directory: {os.getcwd()}")
print("="*70)

## 2️⃣ Mount Google Drive (Option A) - Khuyên dùng

Nếu bạn đã upload data lên Drive, dùng cell này để mount Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Thay đổi path này theo vị trí data của bạn trên Drive
DRIVE_DATA_PATH = '/content/drive/MyDrive/train/'

# Copy data to local (faster access)
print("\n[+] Copying data from Drive to local...")
!cp -r {DRIVE_DATA_PATH}/output /content/
!cp -r {DRIVE_DATA_PATH}/output_safe /content/

# Copy Python files
!cp {DRIVE_DATA_PATH}/*.py /content/

print("✅ Data copied successfully!")

## 2️⃣ Upload Data (Option B) - Upload trực tiếp

Nếu không dùng Drive, uncomment và chạy cell dưới để upload data trực tiếp.

**Lưu ý**: Cần upload 2 file zip:
- `output.zip` (vulnerable samples)
- `output_safe.zip` (safe samples)

In [ ]:
# # Uncomment để upload trực tiếp
# from google.colab import files
# import zipfile

# print("[+] Upload output.zip (vulnerable samples)...")
# uploaded = files.upload()

# print("\n[+] Upload output_safe.zip (safe samples)...")
# uploaded = files.upload()

# # Unzip
# print("\n[+] Extracting files...")
# !unzip -q output.zip
# !unzip -q output_safe.zip

# print("✅ Data extracted!")

## 3️⃣ Install Dependencies

In [ ]:
print("[+] Installing PyTorch...")
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

print("\n[+] Installing Transformers & other libraries...")
!pip install -q transformers==4.35.0
!pip install -q scikit-learn tqdm tensorboard

print("\n✅ Installation complete!")

# Verify installations
import torch
import transformers
from sklearn import __version__ as sklearn_version

print("\n" + "="*70)
print("INSTALLED VERSIONS")
print("="*70)
print(f"PyTorch: {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"Scikit-learn: {sklearn_version}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print("="*70)

## 4️⃣ Upload Python Files

Upload các file .py cần thiết:
- `graphcodebert_tokenizer.py`
- `graphcodebert_dataset.py`
- `graphcodebert_model.py`
- `train_graphcodebert.py`

In [ ]:
# Nếu chưa copy từ Drive, uncomment để upload
# from google.colab import files

# print("[+] Upload Python files (.py)...")
# print("Upload các files: graphcodebert_tokenizer.py, graphcodebert_dataset.py,")
# print("                  graphcodebert_model.py, train_graphcodebert.py")
# uploaded = files.upload()

# print("\n✅ Files uploaded!")

# List uploaded files
print("\n[+] Python files in current directory:")
!ls -lh *.py

## 5️⃣ Verify Data Structure

In [ ]:
import os

print("="*70)
print("VERIFYING DATA STRUCTURE")
print("="*70)

required_dirs = [
    'output/Buffer_Overflow',
    'output/Command_Injection',
    'output/Path_Traversal',
    'output/SQL_Injection',
    'output_safe/Buffer_Overflow',
    'output_safe/Command_Injection',
    'output_safe/Path_Traversal',
    'output_safe/SQL_Injection'
]

all_ok = True
total_files = 0

for dir_path in required_dirs:
    if os.path.isdir(dir_path):
        files = [f for f in os.listdir(dir_path) if f.endswith('.json')]
        count = len(files)
        total_files += count
        print(f"✅ {dir_path:50s} - {count:4d} files")
    else:
        print(f"❌ {dir_path:50s} - NOT FOUND")
        all_ok = False

print("="*70)
if all_ok:
    print(f"✅ All directories found! Total: {total_files} JSON files")
else:
    print("❌ Some directories are missing! Please check your data upload.")
print("="*70)

## 6️⃣ Tokenization

Xử lý và tokenize data thành format GraphCodeBERT.

**Thời gian ước tính**: 5-10 phút (tùy số lượng files)

In [ ]:
from graphcodebert_tokenizer import process_all_data, create_train_val_test_split

print("="*70)
print("STARTING TOKENIZATION")
print("="*70)
print("This may take 5-10 minutes depending on dataset size...\n")

# Process all data
all_data, stats = process_all_data()

if len(all_data) > 0:
    print(f"\n✅ Successfully tokenized {len(all_data)} samples")
    
    # Create train/val/test splits
    print("\n[+] Creating train/val/test splits...")
    splits = create_train_val_test_split(all_data)
    
    print("\n✅ Tokenization complete!")
    print(f"   Train: {len(splits['train'])} samples")
    print(f"   Val:   {len(splits['val'])} samples")
    print(f"   Test:  {len(splits['test'])} samples")
else:
    print("\n❌ No data found! Please check your data structure.")

## 7️⃣ Test Dataset Loading

In [ ]:
from graphcodebert_dataset import load_datasets

print("[+] Testing dataset loading...\n")

dataloaders = load_datasets('processed_graphcodebert', batch_size=8)

if 'train' in dataloaders:
    print("\n✅ Dataset loaded successfully!")
    
    # Get one batch to test
    batch = next(iter(dataloaders['train']))
    
    print("\n[+] Sample batch:")
    print(f"   Input IDs shape:     {batch['input_ids'].shape}")
    print(f"   Attention mask shape: {batch['attention_mask'].shape}")
    print(f"   DFG matrix shape:    {batch['dfg_matrix'].shape}")
    print(f"   Labels shape:        {batch['label'].shape}")
    print(f"   Labels in batch:     {batch['label'].tolist()}")
else:
    print("\n❌ Failed to load dataset!")

## 8️⃣ Training Configuration

Adjust hyperparameters nếu cần (ví dụ: giảm batch_size nếu GPU out of memory)

In [ ]:
# Config for training
CONFIG = {
    'model_type': 'simple',  # 'simple' or 'gnn'
    'model_name': 'microsoft/graphcodebert-base',
    'num_labels': 2,
    'use_dfg': True,
    
    # Training params
    'batch_size': 16,  # Giảm xuống 8 nếu GPU out of memory
    'learning_rate': 2e-5,
    'num_epochs': 10,
    'warmup_steps': 100,
    'max_grad_norm': 1.0,
    'weight_decay': 0.01,
    
    # Paths
    'processed_dir': 'processed_graphcodebert',
    'output_dir': 'models/graphcodebert_vuln_detector',
    'log_dir': 'logs/graphcodebert',
    
    # Device
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'num_workers': 0
}

print("="*70)
print("TRAINING CONFIGURATION")
print("="*70)
for key, value in CONFIG.items():
    print(f"{key:20s}: {value}")
print("="*70)

## 9️⃣ Start Training

**Thời gian ước tính**: 1-2 giờ cho 10 epochs

⚠️ **Lưu ý**: Colab có thể ngắt kết nối sau 12 giờ hoặc nếu idle quá lâu. Hãy theo dõi training!

In [ ]:
from train_graphcodebert import Trainer

print("="*70)
print("STARTING TRAINING")
print("="*70)
print("This will take approximately 1-2 hours...\n")

# Create trainer
trainer = Trainer(CONFIG)

# Start training
trainer.train()

print("\n" + "="*70)
print("✅ TRAINING COMPLETED!")
print("="*70)

## 🔟 View Training Logs with TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs/graphcodebert

## 1️⃣1️⃣ Evaluate on Test Set

In [ ]:
import torch
from graphcodebert_model import create_model
from graphcodebert_dataset import load_datasets
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

print("[+] Loading best model for evaluation...\n")

# Load best model
model = create_model(
    model_type=CONFIG['model_type'],
    num_labels=CONFIG['num_labels'],
    use_dfg=CONFIG['use_dfg']
)

checkpoint = torch.load('models/graphcodebert_vuln_detector/best_model.pt')
model.load_state_dict(checkpoint['model_state_dict'])
model.to(CONFIG['device'])
model.eval()

print("✅ Model loaded!\n")

# Load test data
dataloaders = load_datasets(CONFIG['processed_dir'], batch_size=16)

if 'test' in dataloaders:
    print("[+] Evaluating on test set...\n")
    
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch in dataloaders['test']:
            input_ids = batch['input_ids'].to(CONFIG['device'])
            attention_mask = batch['attention_mask'].to(CONFIG['device'])
            dfg_matrix = batch['dfg_matrix'].to(CONFIG['device'])
            labels = batch['label'].to(CONFIG['device'])
            
            logits = model(input_ids, attention_mask, dfg_matrix=dfg_matrix)
            preds = torch.argmax(logits, dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    # Print results
    print("="*70)
    print("TEST SET RESULTS")
    print("="*70)
    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds, target_names=['Safe', 'Vulnerable']))
    
    print("\nConfusion Matrix:")
    cm = confusion_matrix(all_labels, all_preds)
    print(cm)
    print("\n         Predicted")
    print("         Safe  Vuln")
    print(f"Safe    [{cm[0,0]:4d}  {cm[0,1]:4d}]")
    print(f"Vuln    [{cm[1,0]:4d}  {cm[1,1]:4d}]")
    print("="*70)
else:
    print("❌ Test set not found!")

## 1️⃣2️⃣ Save Model to Drive (Recommended)

In [ ]:
import shutil

print("[+] Compressing model...\n")

# Zip models folder
shutil.make_archive('trained_models', 'zip', 'models/')

print("✅ Model compressed: trained_models.zip\n")

# Copy to Drive
print("[+] Copying to Google Drive...")
!cp trained_models.zip /content/drive/MyDrive/

print("\n✅ Model saved to Google Drive!")
print("   Location: /content/drive/MyDrive/trained_models.zip")

## 1️⃣3️⃣ Download Model (Alternative)

In [ ]:
from google.colab import files

print("[+] Preparing model for download...\n")

# Download compressed model
files.download('trained_models.zip')

print("\n✅ Download started! Check your browser's download folder.")

## 1️⃣4️⃣ Clean Up (Optional)

Giải phóng memory và xóa temporary files nếu cần.

In [ ]:
import gc
import torch

# Clear GPU memory
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Clear Python memory
gc.collect()

print("✅ Memory cleared!")

# Check GPU memory
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated(0) / 1e9
    reserved = torch.cuda.memory_reserved(0) / 1e9
    print(f"\nGPU Memory:")
    print(f"  Allocated: {allocated:.2f} GB")
    print(f"  Reserved:  {reserved:.2f} GB")

---

## 📝 Notes & Tips

### ⚠️ Common Issues:

1. **GPU Out of Memory**:
   - Giảm `batch_size` trong CONFIG từ 16 → 8 hoặc 4
   - Giảm `max_code_length` trong tokenizer

2. **Colab Timeout**:
   - Colab có thể ngắt sau 12h hoặc idle quá lâu
   - Save checkpoints định kỳ (đã implement trong trainer)
   - Có thể resume training từ checkpoint

3. **Data Upload Slow**:
   - Khuyên dùng Google Drive thay vì upload trực tiếp
   - Compress data trước khi upload

### 📊 Expected Performance:
- **Accuracy**: 85-90%
- **F1 Score**: 0.85-0.88
- **Training time**: 1-2 hours (10 epochs)

### 🔧 Customization:
- Change model: `CONFIG['model_type'] = 'gnn'`
- More epochs: `CONFIG['num_epochs'] = 20`
- Different LR: `CONFIG['learning_rate'] = 1e-5`

### 📚 Resources:
- [GraphCodeBERT Paper](https://arxiv.org/abs/2009.08366)
- [Transformers Documentation](https://huggingface.co/docs/transformers)
- [PyTorch Tutorial](https://pytorch.org/tutorials/)

---

**🎉 Happy Training!**